# Ibsen Allusjonsdeteksjon med Gemma
Dette scriptet kjører på Colab med GPU. 
1. Koble til en T4 GPU (Runtime -> Change runtime type -> T4 GPU).
2. Kjør cellen under for å koble til Google Drive.
3. Fyll inn Hugging Face token.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers torch accelerate
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login

# Lim inn din HF token her
login('DIN_HF_TOKEN_HER')

In [ ]:
import sqlite3
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'google/gemma-2-2b'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Laster {model_id} på {device}...')
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device)
print('Modell lastet!')

In [ ]:
DB_PATH = '/content/drive/MyDrive/Colab Notebooks/tei_snippets.db'
conn = sqlite3.connect(DB_PATH)

# 1. Sørg for at tabellen eksisterer (hvis dette er første kjøring)
conn.execute('''
    CREATE TABLE IF NOT EXISTS snippets_surprisal (
        snippet_id TEXT,
        phrase TEXT,
        surprisal_score REAL
    )
''')

# 2. Hent KUN de tekstene som IKKE allerede er ferdig analysert!
query = '''
    SELECT snippet_id, text 
    FROM snippets 
    WHERE text IS NOT NULL AND genre IN ('Dikt', 'Drama')
    AND snippet_id NOT IN (SELECT DISTINCT snippet_id FROM snippets_surprisal)
'''
df = pd.read_sql_query(query, conn)
print(f'Gjenstår {len(df)} tekstsnutter å prosessere.')

In [ ]:
import torch.nn.functional as F

def get_surprising_phrases(text, window_size=6):
    if not text.strip():
        return []
    
    inputs = tokenizer(text, return_tensors='pt').to(device)
    input_ids = inputs['input_ids'][0]
    
    if len(input_ids) < window_size + 1:
        return []
        
    with torch.no_grad():
        outputs = model(**inputs)
        
    logits = outputs.logits[0, :-1, :] 
    labels = input_ids[1:]
    
    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    token_losses = loss_fct(logits, labels)
    
    phrases = []
    for i in range(len(token_losses) - window_size + 1):
        window_loss = token_losses[i : i+window_size].mean().item()
        window_tokens = input_ids[i+1 : i+1+window_size]
        phrase_text = tokenizer.decode(window_tokens, skip_special_tokens=True).strip()
        
        if len(phrase_text) > 5:
            phrases.append({
                "phrase": phrase_text,
                "surprisal_score": round(window_loss, 3)
            })
            
    phrases.sort(key=lambda x: x["surprisal_score"], reverse=True)
    return phrases[:3]

print('Beregner allusjons-fraser...')
all_phrases = []
chunk_size = 500  # Hvor ofte den skal lagre til databasen i Google Drive

for i, (idx, row) in enumerate(df.iterrows()):
    top_phrases = get_surprising_phrases(row['text'])
    for p in top_phrases:
        all_phrases.append({
            "snippet_id": row['snippet_id'],
            "phrase": p["phrase"],
            "surprisal_score": p["surprisal_score"]
        })
        
    # --- LAGRE UNDERVEIS (CHUNKING) ---
    if (i + 1) % chunk_size == 0:
        df_chunk = pd.DataFrame(all_phrases)
        df_chunk.to_sql('snippets_surprisal', conn, if_exists='append', index=False)
        conn.commit()
        print(f'Lagret {i + 1} av {len(df)} snutter til Drive...')
        all_phrases = [] # Tøm listen for neste chunk

# Lagre eventuelle rester på slutten
if all_phrases:
    df_chunk = pd.DataFrame(all_phrases)
    df_chunk.to_sql('snippets_surprisal', conn, if_exists='append', index=False)
    conn.commit()

print('Alt er ferdig analysert og lagret!')
conn.close()
